# Project 6: Employee Engagement Segmentation Dashboard

**Research foundation:**
- **Gallup Q12 Employee Engagement Survey** — 12 questions measuring the psychological conditions employees need to be engaged. Each item rated 1–5.
- **Gallup State of the Global Workplace Report (2023)** — 23% of employees globally are engaged, 59% not engaged, 18% actively disengaged. Cost of disengagement: $8.8 trillion globally (9% of GDP). Manager quality accounts for 70% of team engagement variance. Engaged employees show 81% lower absenteeism and 23% higher profitability.

**Objective:** Segment 900 employees into Gallup's three engagement tiers, analyse Q12 item patterns by department and manager, surface flight risk profiles, and quantify the business impact of disengagement.


## 0. Setup and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

SEGMENT_PALETTE = {
    'Engaged':              '#1D9E75',
    'Not Engaged':          '#EF9F27',
    'Actively Disengaged':  '#E24B4A',
}
SEGMENT_ORDER = ['Engaged', 'Not Engaged', 'Actively Disengaged']

Q12_LABELS = {
    'q01': 'Q01 Expectations clear',
    'q02': 'Q02 Materials / equipment',
    'q03': 'Q03 Do what I do best',
    'q04': 'Q04 Recognition (7 days)',
    'q05': 'Q05 Someone cares about me',
    'q06': 'Q06 Development encouraged',
    'q07': 'Q07 Opinions count',
    'q08': 'Q08 Mission / purpose',
    'q09': 'Q09 Quality commitment',
    'q10': 'Q10 Best friend at work',
    'q11': 'Q11 Progress discussed',
    'q12': 'Q12 Learn and grow',
}
Q12_COLS = list(Q12_LABELS.keys())

df = pd.read_csv('engagement_data.csv', parse_dates=['hire_date'])
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head(3)

## 1. Engagement segmentation overview

Gallup's three-tier segmentation (Gallup State of the Global Workplace, 2023):
- **Engaged (avg ≥ 4.0):** Psychologically committed; enthusiastic contributors.
- **Not Engaged (2.5–3.9):** Doing the minimum required; quiet quitting.
- **Actively Disengaged (< 2.5):** Checked out; may actively undermine colleagues.

Gallup 2023 global benchmarks: 23% / 59% / 18%.

In [ ]:
# Segment distribution
seg_counts = df['engagement_segment'].value_counts().reindex(SEGMENT_ORDER)
seg_pct    = df['engagement_segment'].value_counts(normalize=True).mul(100).reindex(SEGMENT_ORDER).round(1)

benchmarks = {'Engaged': 23.0, 'Not Engaged': 59.0, 'Actively Disengaged': 18.0}

summary = pd.DataFrame({
    'Headcount':         seg_counts,
    'This Dataset (%)':  seg_pct,
    'Gallup 2023 (%)':   pd.Series(benchmarks),
})
print('Engagement Segmentation vs. Gallup 2023 Global Benchmark')
print(summary.to_string())

In [ ]:
# Donut chart — segmentation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Dataset donut
ax = axes[0]
sizes  = [seg_counts[s] for s in SEGMENT_ORDER]
colors = [SEGMENT_PALETTE[s] for s in SEGMENT_ORDER]
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=SEGMENT_ORDER,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.55),
)
for t in autotexts:
    t.set_fontsize(10)
ax.set_title('This Dataset', fontweight='bold')

# Gallup benchmark donut
ax2 = axes[1]
gallup_sizes = [benchmarks[s] for s in SEGMENT_ORDER]
ax2.pie(
    gallup_sizes,
    labels=SEGMENT_ORDER,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.55),
)
for t in ax2.texts:
    t.set_fontsize(10)
ax2.set_title('Gallup 2023 Global Benchmark', fontweight='bold')

plt.suptitle('Employee Engagement Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Q12 score distribution by segment

In [ ]:
# KDE of avg Q12 score by segment
fig, ax = plt.subplots(figsize=(11, 5))

for seg in SEGMENT_ORDER:
    subset = df.loc[df['engagement_segment'] == seg, 'avg_q12_score']
    subset.plot.kde(ax=ax, label=seg, color=SEGMENT_PALETTE[seg], linewidth=2)

ax.axvline(4.0, color='#1D9E75', linestyle='--', linewidth=1.0, label='Engaged threshold (4.0)')
ax.axvline(2.5, color='#E24B4A', linestyle='--', linewidth=1.0, label='Disengaged threshold (2.5)')

ax.set_xlabel('Average Q12 Score')
ax.set_ylabel('Density')
ax.set_title(
    'Q12 Score Distribution by Engagement Segment\n'
    '(Gallup thresholds: Engaged ≥ 4.0 | Actively Disengaged < 2.5)',
    fontweight='bold',
)
ax.legend()
ax.set_xlim(1, 5)
plt.tight_layout()
plt.show()

## 3. Department-level engagement analysis

In [ ]:
# Average Q12 score by department
dept_avg = (
    df.groupby('department')['avg_q12_score']
    .mean()
    .sort_values(ascending=True)
    .round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: avg Q12 by department
ax = axes[0]
colors_dept = ['#E24B4A' if v < 2.5 else '#EF9F27' if v < 4.0 else '#1D9E75' for v in dept_avg.values]
ax.barh(dept_avg.index, dept_avg.values, color=colors_dept, alpha=0.85)
ax.axvline(4.0, color='#1D9E75', linestyle='--', linewidth=1.0)
ax.axvline(2.5, color='#E24B4A', linestyle='--', linewidth=1.0)
for i, (dept, val) in enumerate(dept_avg.items()):
    ax.text(val + 0.02, i, f'{val:.2f}', va='center', fontsize=9)
ax.set_xlabel('Average Q12 Score')
ax.set_title('Average Q12 Score by Department', fontweight='bold')
ax.set_xlim(1, 5.2)

# Right: stacked bar — segment % by department
ax2 = axes[1]
dept_seg = (
    df.groupby(['department', 'engagement_segment'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=SEGMENT_ORDER, fill_value=0)
)
dept_seg_pct = dept_seg.div(dept_seg.sum(axis=1), axis=0) * 100
dept_seg_pct = dept_seg_pct.reindex(dept_avg.index)  # same order

left = np.zeros(len(dept_seg_pct))
for seg in SEGMENT_ORDER:
    ax2.barh(
        dept_seg_pct.index,
        dept_seg_pct[seg],
        left=left,
        label=seg,
        color=SEGMENT_PALETTE[seg],
        alpha=0.85,
    )
    left += dept_seg_pct[seg].values

ax2.set_xlabel('% of Department')
ax2.set_title('Segment Mix by Department', fontweight='bold')
ax2.legend(loc='lower right', fontsize=9)

plt.suptitle('Department Engagement Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Q12 item heatmap — weakest engagement dimensions

Gallup recommends focusing on the bottom 2–3 Q12 items per team because targeted improvement on the weakest items yields the highest engagement lift. Items Q04, Q05, Q06, Q07, and Q11 are manager-heavy — low scores here signal a leadership intervention need.

In [ ]:
# Q12 item averages by department
q12_dept = (
    df.groupby('department')[Q12_COLS]
    .mean()
    .rename(columns=Q12_LABELS)
    .round(3)
)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    q12_dept,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=1.0,
    vmax=5.0,
    center=3.0,
    linewidths=0.4,
    ax=ax,
)
ax.set_title(
    'Q12 Item Average Scores by Department\n'
    '(Manager-heavy items: Q04, Q05, Q06, Q07, Q11 | Green = strong | Red = intervention needed)',
    fontweight='bold',
)
ax.set_ylabel('Department')
ax.set_xlabel('Q12 Item')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Overall weakest Q12 items — ranked
overall_q12 = (
    df[Q12_COLS]
    .mean()
    .rename(Q12_LABELS)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(11, 5))
colors_q12 = ['#E24B4A' if v < 2.5 else '#EF9F27' if v < 3.5 else '#1D9E75' for v in overall_q12.values]
ax.barh(overall_q12.index, overall_q12.values, color=colors_q12, alpha=0.85)
for i, val in enumerate(overall_q12.values):
    ax.text(val + 0.02, i, f'{val:.2f}', va='center', fontsize=9)
ax.axvline(3.5, color='#444441', linestyle='--', linewidth=1.0, label='Attention threshold (3.5)')
ax.set_xlabel('Average Score (1-5)')
ax.set_title(
    'Q12 Items Ranked by Average Score — Weakest to Strongest\n'
    '(Gallup: focus interventions on the bottom 2-3 items per team)',
    fontweight='bold',
)
ax.legend()
ax.set_xlim(1, 5.3)
plt.tight_layout()
plt.show()

## 5. Manager quality and team engagement

Gallup's State of the Global Workplace (2023) found that manager quality explains approximately 70% of variance in team engagement. This section validates that pattern in the dataset and surfaces the managers whose teams are most at risk.

In [ ]:
# Manager-level aggregation
mgr_summary = (
    df.groupby('manager_id')
    .agg(
        team_size=('employee_id', 'count'),
        team_avg_q12=('avg_q12_score', 'mean'),
        manager_quality=('manager_quality', 'mean'),
        flight_risk_count=('flight_risk_flag', 'sum'),
        avg_absence_days=('absence_days', 'mean'),
    )
    .query('team_size >= 3')
    .reset_index()
)

# Correlation: manager quality vs. team avg Q12
r, p = stats.pearsonr(mgr_summary['manager_quality'], mgr_summary['team_avg_q12'])
print(f'Pearson correlation (manager quality vs. team Q12 avg): r = {r:.3f}, p = {p:.4f}')
print(f'R-squared: {r**2:.3f} (Gallup benchmark: ~0.70)')

In [ ]:
# Scatter: manager quality vs. team engagement
fig, ax = plt.subplots(figsize=(10, 5))

scatter = ax.scatter(
    mgr_summary['manager_quality'],
    mgr_summary['team_avg_q12'],
    c=mgr_summary['flight_risk_count'],
    cmap='RdYlGn_r',
    s=60,
    alpha=0.75,
    edgecolors='white',
    linewidths=0.4,
)

# Regression line
m_fit, b_fit = np.polyfit(mgr_summary['manager_quality'], mgr_summary['team_avg_q12'], 1)
x_range = np.linspace(mgr_summary['manager_quality'].min(), mgr_summary['manager_quality'].max(), 100)
ax.plot(x_range, m_fit * x_range + b_fit, color='#444441', linewidth=1.5, linestyle='--')

plt.colorbar(scatter, ax=ax, label='Flight Risk Headcount')

ax.set_xlabel('Manager Quality Score (0-1)')
ax.set_ylabel('Team Average Q12 Score')
ax.set_title(
    f'Manager Quality vs. Team Engagement  (r = {r:.2f}, R² = {r**2:.2f})\n'
    'Gallup: manager quality explains ~70% of engagement variance',
    fontweight='bold',
)
plt.tight_layout()
plt.show()

## 6. Tenure-engagement curve

Gallup documents a honeymoon effect for new hires and a bifurcation at long tenure: highly tenured employees are either deeply committed or checked out.

In [ ]:
# Bin tenure into bands
bins   = [0, 0.5, 1.0, 2.0, 5.0, 10.0, 100]
labels = ['0-6 months', '6-12 months', '1-2 years', '2-5 years', '5-10 years', '10+ years']

df['tenure_band'] = pd.cut(df['years_tenure'], bins=bins, labels=labels, right=False)

tenure_eng = (
    df.groupby('tenure_band', observed=True)['avg_q12_score']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
tenure_eng['se'] = tenure_eng['std'] / np.sqrt(tenure_eng['count'])

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    tenure_eng['tenure_band'],
    tenure_eng['mean'],
    marker='o', color='#378ADD', linewidth=2, markersize=7,
)
ax.fill_between(
    tenure_eng['tenure_band'],
    tenure_eng['mean'] - tenure_eng['se'],
    tenure_eng['mean'] + tenure_eng['se'],
    alpha=0.15, color='#378ADD',
)

ax.axhline(4.0, color='#1D9E75', linestyle='--', linewidth=1.0, label='Engaged threshold')
ax.axhline(2.5, color='#E24B4A', linestyle='--', linewidth=1.0, label='Disengaged threshold')

ax.set_xlabel('Tenure Band')
ax.set_ylabel('Average Q12 Score')
ax.set_title(
    'Engagement by Tenure Band — Honeymoon Effect and Long-Tenure Bifurcation\n'
    '(Gallup: new hire optimism peaks before 6 months; long-tenure employees bifurcate)',
    fontweight='bold',
)
ax.legend()
ax.set_ylim(1, 5)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 7. Flight risk analysis

In [ ]:
# Flight risk overview
total_flight = df['flight_risk_flag'].sum()
flight_pct   = df['flight_risk_flag'].mean() * 100
print(f'Total flight risk employees : {total_flight:,} ({flight_pct:.1f}% of workforce)')

# By department
flight_dept = (
    df.groupby('department')
    .agg(headcount=('employee_id', 'count'), flight_risk=('flight_risk_flag', 'sum'))
    .assign(flight_risk_pct=lambda x: (x['flight_risk'] / x['headcount'] * 100).round(1))
    .sort_values('flight_risk_pct', ascending=False)
)
print('\nFlight risk by department:')
print(flight_dept.to_string())

In [ ]:
# Scatter: intent to stay vs. Q12 score, coloured by flight risk
fig, ax = plt.subplots(figsize=(10, 5))

for is_risk, grp in df.groupby('flight_risk_flag'):
    label  = 'Flight Risk' if is_risk else 'Retained'
    color  = '#E24B4A'     if is_risk else '#378ADD'
    marker = 'X'           if is_risk else 'o'
    ax.scatter(
        grp['avg_q12_score'],
        grp['intent_to_stay'],
        label=label,
        color=color,
        marker=marker,
        alpha=0.5,
        s=25,
    )

ax.axvline(2.5, color='#E24B4A', linestyle='--', linewidth=1.0)
ax.axhline(2.0, color='#E24B4A', linestyle='--', linewidth=1.0)
ax.set_xlabel('Average Q12 Score')
ax.set_ylabel('Intent to Stay (1-5)')
ax.set_title(
    'Flight Risk: Q12 Score vs. Intent to Stay\n'
    '(Bottom-left quadrant = highest urgency retention cases)',
    fontweight='bold',
)
ax.legend()
plt.tight_layout()
plt.show()

## 8. Business impact — absenteeism and performance by segment

Gallup's meta-analysis quantifies the business cost of disengagement: 81% higher absenteeism and 23% lower profitability for actively disengaged employees vs. engaged peers.

In [ ]:
# Business outcomes by segment
outcomes = (
    df.groupby('engagement_segment')
    .agg(
        headcount=('employee_id', 'count'),
        avg_q12=('avg_q12_score', 'mean'),
        avg_absence=('absence_days', 'mean'),
        avg_performance=('performance_rating', 'mean'),
        avg_intent=('intent_to_stay', 'mean'),
        flight_risk_pct=('flight_risk_flag', 'mean'),
    )
    .reindex(SEGMENT_ORDER)
    .round(2)
)

engaged_abs    = outcomes.loc['Engaged', 'avg_absence']
disengaged_abs = outcomes.loc['Actively Disengaged', 'avg_absence']
absence_gap    = (disengaged_abs - engaged_abs) / disengaged_abs * 100

print(outcomes.to_string())
print(f'\nAbsenteeism reduction (engaged vs. disengaged): {absence_gap:.0f}%  (Gallup benchmark: 81%)')

In [ ]:
# Side-by-side outcome comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics = [
    ('avg_absence',     'Avg Absence Days',       False),
    ('avg_performance', 'Avg Performance Rating', True),
    ('avg_intent',      'Avg Intent to Stay',     True),
]

for ax, (metric, title, good_is_high) in zip(axes, metrics):
    values = outcomes[metric].reindex(SEGMENT_ORDER)
    colors = [SEGMENT_PALETTE[s] for s in SEGMENT_ORDER]
    bars   = ax.bar(range(len(SEGMENT_ORDER)), values, color=colors, width=0.55, alpha=0.85)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f'{val:.1f}',
            ha='center', va='bottom', fontsize=10,
        )

    ax.set_xticks(range(len(SEGMENT_ORDER)))
    ax.set_xticklabels(['Engaged', 'Not\nEngaged', 'Actively\nDisengaged'], fontsize=9)
    ax.set_title(title, fontweight='bold')

plt.suptitle(
    'Business Outcomes by Engagement Segment\n'
    '(Gallup: engaged employees show 81% lower absenteeism, 23% higher profitability)',
    fontsize=12,
    fontweight='bold',
)
plt.tight_layout()
plt.show()

## 9. eNPS analysis

In [ ]:
# eNPS calculation
promoters   = (df['enps_segment'] == 'Promoter').sum()
detractors  = (df['enps_segment'] == 'Detractor').sum()
passives    = (df['enps_segment'] == 'Passive').sum()
n           = len(df)

enps = (promoters - detractors) / n * 100

print(f'Promoters  (score 9-10): {promoters:,} ({promoters/n*100:.1f}%)')
print(f'Passives   (score 7-8):  {passives:,}  ({passives/n*100:.1f}%)')
print(f'Detractors (score 0-6):  {detractors:,} ({detractors/n*100:.1f}%)')
print(f'\neNPS Score: {enps:.1f}')
print('(Gallup: positive eNPS is a leading indicator of engagement health)')

## 10. Key findings summary

In [ ]:
engaged_pct    = seg_pct.get('Engaged', 0)
disengaged_pct = seg_pct.get('Actively Disengaged', 0)

print('=' * 65)
print('ENGAGEMENT AUDIT — KEY FINDINGS SUMMARY')
print('=' * 65)
print(f"""
1. SEGMENTATION:
   Engaged:              {engaged_pct:.1f}%  (Gallup global: 23%)
   Not Engaged:          {seg_pct.get('Not Engaged', 0):.1f}%  (Gallup global: 59%)
   Actively Disengaged:  {disengaged_pct:.1f}%  (Gallup global: 18%)

2. MANAGER EFFECT:
   Manager quality vs. team Q12 avg correlation: r = {r:.2f}
   R-squared: {r**2:.2f} — consistent with Gallup's 70% manager variance finding.
   Bottom-quartile managers carry disproportionate flight risk.

3. ABSENTEEISM IMPACT:
   Engaged avg absence:            {engaged_abs:.1f} days
   Actively Disengaged avg absence: {disengaged_abs:.1f} days
   Reduction if all disengaged employees became engaged: {absence_gap:.0f}%
   Gallup benchmark: 81% lower absenteeism for engaged employees.

4. FLIGHT RISK:
   Employees flagged:  {total_flight:,} ({flight_pct:.1f}% of workforce)
   Concentrated in:    Lowest Q12 scoring departments + bottom manager quartile.

5. eNPS: {enps:.1f}
   Gallup: positive eNPS correlates with engaged workforce.

RECOMMENDED ACTIONS (Gallup framework):
  - Manager coaching for bottom-quartile managers (Q04/Q05/Q06 intervention)
  - Retention conversations with all flight-risk employees within 30 days
  - Focus pulse survey on weakest 2-3 Q12 items per department
  - Set an Engaged % target of >= 30% within 12 months
""")
print('=' * 65)